# Draw — Treino completo SegFormer-B0 (GPU, 3.000 cenas)

**Versão de treino completo, pronta para colar no Kaggle.**
Modelo SegFormer-B0, 4 classes (0=background, 1=paper, 2=hand, 3=ink),
dataset sintético de 3.000 cenas em 8 partes, pesos CE `[1.0, 1.5, 10.0, 10.0]`
(recalibrados do split de treino, commit `86bbed3`, branch `feature/main-integracao`),
30 épocas da config com early stopping, batch 4.

**Inputs obrigatórios do kernel (todos como Add Input, antes de rodar):**
`guzzfort/draw-synthetic-v1-part1` até `part8`,
`guzzfort/draw-training-code` (versão nova, com os pesos),
`danieldelro/freihand`, `guzzfort/mit-b0-cache` (v2, com `refs/main` sem newline).

**Este notebook NÃO faz `pip install` nem `git clone`.**
Se rodar sem GPU, a célula de ambiente PARA — o treino completo em CPU é inviável.
Ordem: ambiente → cache de pesos → código → junção das 8 partes → FreiHAND →
config/pixels → treino → artefatos. Não pule a junção: sem ela, o treino usaria
só a parte 1 (400 imagens).

## 1. Verificar o ambiente (GPU obrigatória)

Checa torch, transformers, cv2 e numpy, imprime a GPU e **trava se não houver CUDA**.
`onnxruntime` é opcional: sem ele, o `train.py` pula a validação ONNX e
`onnx_validation` vem `null` no `metrics.json` — o ONNX ainda é gerado.

In [ ]:
# Verificação do ambiente — SEM instalar pacotes, SEM download.
# GPU é OBRIGATÓRIA neste notebook de treino completo: sem CUDA, para aqui.
import importlib

REQUIRED = ["torch", "transformers", "cv2", "numpy"]
OPTIONAL = {
    "albumentations": "treino sem augmentation",
    "onnx": "exportação ONNX pode falhar sem ele",
    "onnxruntime": "validação ONNX será pulada pelo train.py",
}
missing = []
for name in REQUIRED:
    try:
        mod = importlib.import_module(name)
        print(f"{name}: OK ({getattr(mod, '__version__', 'sem versão')})")
    except ImportError:
        missing.append(name)
        print(f"{name}: AUSENTE (obrigatório)")
if missing:
    raise RuntimeError(
        "Dependência(s) ausente(s) no ambiente do kernel: " + ", ".join(missing) + ". "
        "Este notebook NÃO instala pacotes (modo offline)."
    )
for name, effect in OPTIONAL.items():
    try:
        importlib.import_module(name)
        print(f"{name}: OK (opcional)")
    except ImportError:
        print(f"{name}: ausente (opcional) -> {effect}")

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "SEM GPU NESTE KERNEL (cuda: False). O treino completo de 30 épocas em "
        "3.000 cenas é inviável em CPU. Pare aqui: crie o notebook na interface "
        "web com Accelerator = GPU T4 e rode de novo — não lance o treino em CPU."
    )
print("GPU:", torch.cuda.get_device_name(0))


## 2. Cache de pesos mit-b0 (offline)

Procura o cache do `nvidia/mit-b0` (dataset `guzzfort/mit-b0-cache` v2) nos
**dois layouts de montagem** do Kaggle e, se achar, fixa `HF_HUB_CACHE` e liga
o modo offline **antes** de qualquer `from_pretrained` (o `train.py` roda como
subprocesso e herda estas variáveis). Sem o cache, o primeiro `from_pretrained`
tentaria baixar do HuggingFace e falharia sem internet.

In [ ]:
# Cache dos pesos pré-treinados nvidia/mit-b0 — ambos os layouts do Kaggle.
import glob as _glob
import os

_hub_candidates = []
for _pat in ("/kaggle/input/*/hub", "/kaggle/input/datasets/*/*/hub", "/kaggle/input/datasets/*/*/*/hub"):
    _hub_candidates += [d for d in _glob.glob(_pat) if os.path.isdir(os.path.join(d, "models--nvidia--mit-b0"))]
if _hub_candidates:
    os.environ["HF_HUB_CACHE"] = _hub_candidates[0]
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("cache de pesos nvidia/mit-b0: encontrado em", _hub_candidates[0], "| modo offline ligado")
else:
    raise RuntimeError(
        "Cache de pesos nvidia/mit-b0 NÃO encontrado. Anexe o dataset "
        "guzzfort/mit-b0-cache (v2) como input do kernel antes de rodar."
    )


## 3. Localizar o código de treino (sem clone)

O código `training/` vem como **input do kernel** (`guzzfort/draw-training-code`),
não do GitHub. A célula procura nos dois layouts do Kaggle e **trava se os pesos
da config não forem os recalibrados** `[1.0, 1.5, 10.0, 10.0]` — se vier a config
antiga `[1.0, 2.0, 3.0, 10.0]`, é a versão errada do dataset de código.

In [ ]:
# Localização do código training/ — SEM clonagem do repo (modo offline).
# Layouts do Kaggle (PROVADO no reteste): inputs ficam tanto em
# /kaggle/input/<slug>/ quanto em /kaggle/input/datasets/<owner>/<slug>/.
# Por isso procuramos nos DOIS layouts; procurar só no antigo falhava.
from pathlib import Path

code_candidates = []
code_candidates += sorted(Path("/kaggle/input").glob("*/training"))
code_candidates += sorted(Path("/kaggle/input").glob("*/draw/training"))
code_candidates += sorted(Path("/kaggle/input/datasets").glob("*/*/training"))
code_candidates += sorted(Path("/kaggle/input/datasets").glob("*/*/draw/training"))
code_candidates += sorted(Path("/kaggle/input/datasets").glob("*/*/*/training"))
code_candidates.append(Path("/kaggle/working/draw/training"))

REPO = None
for cand in dict.fromkeys(code_candidates):
    if (cand / "train.py").exists():
        REPO = cand.parent
        break
if REPO is None:
    raise RuntimeError(
        "Código training/ NÃO encontrado em nenhum input do kernel. "
        "Anexe o dataset de código com training/train.py + training/config.json "
        "como input do kernel (Add Input). Este notebook NÃO clona o repositório. "
        "Caminhos procurados (layout antigo e novo): /kaggle/input/*/training, "
        "/kaggle/input/*/draw/training, /kaggle/input/datasets/*/*/training, "
        "/kaggle/input/datasets/*/*/draw/training, /kaggle/working/draw/training."
    )
print("repo (código):", REPO, "| train.py:", (REPO / "training" / "train.py").exists())

# Trava de pesos: a versão nova do draw-training-code tem que trazer os pesos
# recalibrados do synthetic-v1 (commit 86bbed3). Config antiga = dataset desatualizado.
import json as _json
_cfg = _json.load(open(REPO / "training" / "config.json"))
_pesos = _cfg["loss"]["cross_entropy_class_weights"]
print("pesos da config:", _pesos)
assert _pesos == [1.0, 1.5, 10.0, 10.0], (
    f"Pesos errados na config: {_pesos}. Esperado [1.0, 1.5, 10.0, 10.0]. "
    "Suba a versão nova do guzzfort/draw-training-code antes de treinar."
)
print("pesos OK: [1.0, 1.5, 10.0, 10.0]")


## 4. Junção das 8 partes do dataset (obrigatória, antes do treino)

Copia as 8 partes (`draw-synthetic-v1-part1..8`) para um único dataset em
`/kaggle/working/dataset/synthetic` e **trava se qualquer parte faltar ou se as
contagens não fecharem em 2400/300/300** (imagens E máscaras, com pareamento
por amostra e `split_manifest.json` da parte 1).
**Depois da junção, `DATA` aponta para o diretório reunido** — a autodetectação
do notebook antigo pegaria só a parte 1 (400 imagens de treino). Não rode o
treino sem esta célula.

In [ ]:
# Junção das 8 partes — ambos os layouts de montagem do Kaggle.
import shutil
from pathlib import Path

out = Path("/kaggle/working/dataset/synthetic")
found = []
for n in range(1, 9):
    cands = [
        Path(f"/kaggle/input/draw-synthetic-v1-part{n}/dataset/synthetic"),
        Path(f"/kaggle/input/datasets/guzzfort/draw-synthetic-v1-part{n}/dataset/synthetic"),
    ]
    src = next((c for c in cands if c.is_dir()), None)
    assert src is not None, f"input ausente para a parte {n}: {cands}"
    found.append(src)
    for f in sorted(src.rglob("*")):
        if f.is_file():
            dst = out / f.relative_to(src)
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(f, dst)
print("partes encontradas:", len(found))

# Conferência obrigatória — tem que fechar 2400/300/300.
for split, esperado in (("train", 2400), ("val", 300), ("test", 300)):
    imgs = list((out / split / "images").glob("*.png"))
    masks = list((out / split / "masks").glob("*.png"))
    assert len(imgs) == esperado, f"{split}: {len(imgs)} imagens, esperado {esperado}"
    assert len(masks) == esperado, f"{split}: {len(masks)} máscaras, esperado {esperado}"
    for img in imgs[:3]:
        assert (out / split / "masks" / img.name).exists(), f"máscara ausente: {img.name}"
    print(split, len(imgs), "imagens +", len(masks), "máscaras OK")
assert (out / "split_manifest.json").exists(), "split_manifest.json ausente (vem da parte 1)"
print("split_manifest.json OK")

# O treino DEVE usar o diretório reunido, não um input isolado (parte 1).
DATA = out
print("DATA =", DATA)


## 5. Checagem do FreiHAND (de verdade, não só listagem)

Antes do treino, confirme o que a versão do Kaggle realmente contém: máscara
real de mão em `training/mask/`, com dimensão, limiar 127 e cobertura.
**Se `training/mask/` faltar, esta célula levanta erro e o notebook para aqui**
— não gere rótulos inventados de mão.

In [ ]:
# Checagem real do FreiHAND — PARA com erro claro se training/mask/ faltar.
# Layouts do Kaggle: /kaggle/input/<slug>/... e /kaggle/input/datasets/<owner>/<slug>/...
import cv2
import numpy as np
from pathlib import Path

mask_dirs = sorted(Path("/kaggle/input").glob("*/training/mask"))
mask_dirs += sorted(Path("/kaggle/input").glob("*/*/training/mask"))
mask_dirs += sorted(Path("/kaggle/input/datasets").glob("*/*/training/mask"))
mask_dirs += sorted(Path("/kaggle/input/datasets").glob("*/*/*/training/mask"))
mask_dirs = [d for d in dict.fromkeys(mask_dirs) if d.is_dir()]
if not mask_dirs:
    raise RuntimeError(
        "FreiHAND: training/mask/ NÃO encontrado em nenhum input do kernel. "
        "Anexe o dataset danieldelro/freihand como input do kernel (Add Input). "
        "Sem máscara real de mão, NÃO prossiga: documente o problema e troque a fonte — "
        "nunca invente rótulos de mão. Caminhos procurados (layout antigo e novo): "
        "/kaggle/input/*/training/mask, /kaggle/input/datasets/*/*/training/mask."
    )
mask_dir = mask_dirs[0]
print("FreiHAND training/mask/:", mask_dir)
mask_files = sorted(mask_dir.glob("*.jpg")) + sorted(mask_dir.glob("*.png"))
if not mask_files:
    raise RuntimeError(f"FreiHAND: nenhum arquivo de máscara em {mask_dir}.")
print(f"exemplo de máscara: {mask_files[0].name} (primeiras {min(5, len(mask_files))} serão amostradas)")
coverages = []
for mf in mask_files[:5]:
    mask = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"FreiHAND: não consegui ler a máscara {mf}.")
    if mask.ndim != 2:
        raise RuntimeError(f"FreiHAND: máscara {mf} não é 2D (shape {mask.shape}).")
    hand = mask > 127  # regra de uso: binarizar com limiar 127, nunca usar o JPEG cru
    cov = float(hand.mean())
    coverages.append(cov)
    rgb = mask_dir.parent / "rgb" / mf.name
    print(f"  {mf.name}: shape {mask.shape}, cobertura mão (limiar 127) {cov:.4f}, RGB pareado: {'OK' if rgb.exists() else 'AUSENTE'}")
if all(c == 0.0 for c in coverages):
    raise RuntimeError("FreiHAND: todas as máscaras amostradas têm cobertura 0 — nenhuma mão detectada. Não prossiga sem investigar.")
if all(c >= 0.99 for c in coverages):
    raise RuntimeError("FreiHAND: todas as máscaras amostradas são quase 100% mão — formato inesperado. Não prossiga sem investigar.")
print("FreiHAND OK para checagem: training/mask/ existe, máscara 2D, limiar 127, cobertura de mão plausível.")


## 6. Config e contagem de pixels

Lê `training/config.json` e conta pixels por classe no treino reunido.
Compare com os pesos da config (`[1.0, 1.5, 10.0, 10.0]`): os pesos calculados
aqui são só referência — o `train.py` usa os da config. Nunca ajuste peso
olhando o split de teste.

In [ ]:
import sys, json
sys.path.insert(0, str(REPO))
from training.train import load_config, pair_images_masks, pixel_counts, compute_class_weights

cfg = load_config(REPO / "training" / "config.json")
print("modelo:", cfg["model"]["architecture"], cfg["model"]["backbone"])
print("classes:", cfg["model"]["classes"])
print("loss:", cfg["loss"]["formula"])
pairs = pair_images_masks(DATA / "train")
counts = pixel_counts(pairs)
print("pixels por classe (treino):", counts.tolist())
print("pesos calculados (referência):", [round(w, 2) for w in compute_class_weights(counts)])
print("pesos da config (usados):", cfg["loss"]["cross_entropy_class_weights"])


## 7. Treino completo

Chama `training/train.py`, que é a única fonte de verdade do treino (modelo,
loss, métricas, ONNX). O notebook é só um invólucro reproduzível em cima dele.
`EPOCHS = None` usa a config: **30 épocas, early stopping com paciência 5 em
`val_mean_iou`**. `--batch-size 4` porque o código roda em fp32 sem autocast e
batch 8 é o principal risco de OOM numa T4 — não reduza a resolução, traço fino
depende dela. Tempo estimado numa T4: da ordem de 3–4 h se rodar as 30 épocas.

In [ ]:
# Treino completo — EPOCHS = None (30 da config), batch 4.
EPOCHS = None  # None = usa a config (30 épocas); NÃO troque para 2 aqui (isso era o teste)

import subprocess, sys
cmd = [
    sys.executable,
    str(REPO / "training" / "train.py"),
    "--config", str(REPO / "training" / "config.json"),
    "--data-dir", str(DATA),  # DATA = /kaggle/working/dataset/synthetic, da célula de junção
    "--out-dir", "/kaggle/working/out",
    "--batch-size", "4",
]
if EPOCHS is not None:
    cmd += ["--epochs", str(EPOCHS)]
print("comando:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 8. Métricas e artefatos

`metrics.json` traz IoU, Dice, precision e recall por classe, a métrica de ink
com tolerância (2–3 px) e a validação ONNX. **Nunca reporte só accuracy global:**
com fundo dominante, ela mente. Ao fim, baixe todos os artefatos listados abaixo
— sem eles, o treino não está entregue.

In [ ]:
import json
from pathlib import Path

out_dir = Path("/kaggle/working/out")
metrics_path = out_dir / "metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(
        "metrics.json não encontrado em /kaggle/working/out. O treino não completou "
        "ou salvou em outro lugar — verifique a célula de treino antes de ler métricas."
    )
m = json.load(open(metrics_path))
print("val_mean_iou:", round(m["mean_iou"], 4))
for cls in ("background", "paper", "hand", "ink"):
    d = m[cls]
    print(f"{cls:11s} IoU {d['iou']:.3f}  Dice {d['dice']:.3f}  P {d['precision']:.3f}  R {d['recall']:.3f}")
print("validação ONNX:", m.get("onnx_validation"))
print()
print("artefatos esperados (baixar todos ao fim):")
for name in ("metrics.json", "classes.json", "training_config.json", "split_manifest.json",
             "draw_segmentation_v1.onnx"):
    p = out_dir / name
    print(f"  {'OK ' if p.exists() else 'FALTA'} {name}" + (f" ({p.stat().st_size} bytes)" if p.exists() else ""))
ckpt = out_dir / "checkpoints" / "best.pt"
print(f"  {'OK ' if ckpt.exists() else 'FALTA'} checkpoints/best.pt" + (f" ({ckpt.stat().st_size} bytes)" if ckpt.exists() else ""))
print("todos os arquivos em out:", sorted(p.name for p in out_dir.iterdir()))


## 9. Interface ONNX (para integrar em `draw/neural.py` depois)

- **Input:** `image`, tensor float32 `[1, 3, 512, 512]`, RGB, normalização ImageNet
  (mean 0.485/0.456/0.406, std 0.229/0.224/0.225) — idêntica ao contrato atual de `draw/neural.py`.
- **Output:** `logits`, `[1, 4, 512, 512]`, sem softmax embutido.
- Classes: 0=background, 1=paper, 2=hand, 3=ink. A integração aplica argmax:
  `paper_mask = (argmax == 1)`, `hand_mask = (argmax == 2)`, `ink_mask = (argmax == 3)`.
  `paper_presence` é **derivado** da máscara (fração de pixels paper acima do limiar),
  não uma cabeça auxiliar. O contrato antigo de 2 canais NÃO foi simplificado para
  caber no modelo novo — a ligação só acontece com o modelo final validado.